# Customer Segmentation (RFM)

**InsightForge Business Analytics Studio — Python Analyst Toolkit**

## Purpose
Segment customers by **R**ecency, **F**requency, and **M**onetary value using order history, so the business can target retention/win-back/VIP campaigns differently by segment.

## Methodology
For each customer:
- **Recency** = days between a reference date (defaults to the latest order date in the dataset) and the customer's most recent order.
- **Frequency** = total number of orders placed.
- **Monetary** = total revenue across all of the customer's orders.

Each dimension is scored 1-4 using **explicit, configurable thresholds** (not quantile/percentile binning). This was a deliberate design choice: percentile-based RFM scoring shifts meaning depending on how many customers/orders happen to be in the dataset, which makes the segment definitions hard to reason about or compare across runs. Fixed thresholds are transparent and stable.

### Default thresholds (see `analysis_utils.DEFAULT_RFM_THRESHOLDS`)

| Score | Recency (days, lower=better) | Frequency (orders) | Monetary (revenue $) |
|---|---|---|---|
| 4 | <= 30 | >= 5 | >= 500 |
| 3 | <= 90 | >= 3 | >= 200 |
| 2 | <= 180 | >= 2 | >= 50 |
| 1 | > 180 | < 2 | < 50 |

The three scores are summed (range 3-12) and mapped to a segment label:
`Champions` (10-12), `Loyal Customers` (8-9), `Potential Loyalists` (6-7), `At Risk` (4-5), `Lost` (3).

## Assumptions
- Monetary value uses `revenue`, not `profit` -- a customer who buys a lot of low-margin product still counts as high monetary value here. Swap to `profit` if your business cares more about profitability than topline spend.
- These thresholds are illustrative defaults tuned for this toolkit's synthetic order volumes and price points. Real businesses should recalibrate them (pass a custom `thresholds` dict to `compute_rfm`) to match their own order cadence and typical order value.

## Limitations
- RFM is a simple heuristic segmentation, not a predictive churn model.
- A customer with exactly one large order can score as highly as a genuinely loyal repeat customer; consider frequency-weighting more heavily if that matters for your business.
- Segment boundaries are threshold-based 'cliffs' -- a customer one day of recency past a boundary is scored identically to one 90 days past it.

In [1]:
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

TOOLKIT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, TOOLKIT_DIR)

from clean_data import load_csv, clean_dataframe
from analysis_utils import compute_rfm, DEFAULT_RFM_THRESHOLDS, DEFAULT_SEGMENT_BANDS

OUTPUT_DIR = os.path.join(TOOLKIT_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

orders_path = os.path.join(TOOLKIT_DIR, "sample_data", "orders.csv")
raw_orders = load_csv(orders_path)
orders, cleaning_summary = clean_dataframe(raw_orders)
print(cleaning_summary)

Data Cleaning Summary

- Removed 11 exact duplicate row(s) (kept first occurrence).
- No invalid negative values found in quantity/revenue/cost/discount/rating.
- Missing values detected (left as-is; not auto-filled to avoid inventing data):
    - 'campaign': 3831 missing (60.6% of rows)
    - 'region': 6 missing (0.1% of rows)
    - 'marketingSource': 4 missing (0.1% of rows)
    - 'discount': 3 missing (0.0% of rows)
    - 'channel': 2 missing (0.0% of rows)
- Statistical outliers detected via IQR method (reported only, not removed):
    - 'quantity': 201 value(s) outside [-0.5, 3.5]
    - 'revenue': 580 value(s) outside [-39.06, 185.38]
    - 'cost': 507 value(s) outside [-20.03, 94.18]
    - 'profit': 442 value(s) outside [-30.52, 105.58]
    - 'discount': 615 value(s) outside [-0.08, 0.12]
  Outliers are left in the cleaned data by default -- in real business data these are often legitimate large orders, not errors. Review manually before excluding.

Rows before cleaning: 6326
Row

In [2]:
required_cols = {"customerId", "date", "revenue"}
if not required_cols.issubset(orders.columns):
    print(f"RFM segmentation unavailable: missing column(s) {sorted(required_cols - set(orders.columns))}")
    rfm = None
else:
    rfm = compute_rfm(orders, thresholds=DEFAULT_RFM_THRESHOLDS, segment_bands=DEFAULT_SEGMENT_BANDS)
    display(rfm.head(10))

,customerId,frequency,monetary,recencyDays,rScore,fScore,mScore,rfmScore,segment
0,CUST00001,3,349.12,122,2,3,3,8,Loyal Customers
1,CUST00002,3,235.44,30,4,3,3,10,Champions
2,CUST00003,3,216.74,30,4,3,3,10,Champions
3,CUST00004,1,29.06,490,1,1,1,3,Lost
4,CUST00005,1,29.06,415,1,1,1,3,Lost
5,CUST00006,3,178.36,96,2,3,2,7,Potential Loyalists
6,CUST00007,10,1167.04,20,4,4,4,12,Champions
7,CUST00008,3,225.70,102,2,3,3,8,Loyal Customers
8,CUST00009,2,141.99,311,1,2,2,5,At Risk
9,CUST00010,1,60.27,503,1,1,2,4,At Risk


## Segment Summary

In [3]:
if rfm is not None:
    segment_summary = (
        rfm.groupby("segment")
        .agg(customers=("customerId", "count"), totalRevenue=("monetary", "sum"), avgRecencyDays=("recencyDays", "mean"))
        .sort_values("totalRevenue", ascending=False)
    )
    segment_summary.to_csv(os.path.join(OUTPUT_DIR, "02_rfm_segment_summary.csv"))
    rfm.to_csv(os.path.join(OUTPUT_DIR, "02_rfm_customer_detail.csv"), index=False)
    print("Saved output/02_rfm_segment_summary.csv and output/02_rfm_customer_detail.csv")
    display(segment_summary)

Saved output/02_rfm_segment_summary.csv and output/02_rfm_customer_detail.csv


,customers,totalRevenue,avgRecencyDays
segment,,,
Champions,487,310563.88,29.338809
Loyal Customers,493,142807.48,70.847870
Potential Loyalists,477,87267.80,136.958071
At Risk,515,54851.46,274.937864
Lost,111,3251.02,338.756757


In [4]:
if rfm is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    segment_summary["customers"].sort_values().plot(kind="barh", ax=axes[0], color="#2a5599")
    axes[0].set_title("Customers per Segment")
    axes[0].set_xlabel("Customers")

    segment_summary["totalRevenue"].sort_values().plot(kind="barh", ax=axes[1], color="#3c9c6a")
    axes[1].set_title("Total Revenue per Segment")
    axes[1].set_xlabel("Revenue ($)")

    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "02_segment_chart.png"), dpi=150)
    plt.close(fig)
    print("Saved output/02_segment_chart.png")

Saved output/02_segment_chart.png


## Summary
This notebook wrote `02_rfm_segment_summary.csv`, `02_rfm_customer_detail.csv`, and `02_segment_chart.png` to `output/`.

Revisit the thresholds in the Methodology section above before trusting this segmentation on your own data -- they were tuned for this toolkit's synthetic order volumes, not any particular real business.